In [ ]:
import os
import xml.etree.ElementTree as ET
import csv

def parse_voc_xml(file_path):
    tree = ET.parse(file_path)
    root = tree.getroot()

    image_filename = root.find('filename').text
    image_width = int(root.find('size/width').text)
    image_height = int(root.find('size/height').text)
    
    objects = []
    for obj in root.findall('object'):
        name = obj.find('name').text
        xmin = int(obj.find('bndbox/xmin').text)
        ymin = int(obj.find('bndbox/ymin').text)
        xmax = int(obj.find('bndbox/xmax').text)
        ymax = int(obj.find('bndbox/ymax').text)

        objects.append({
            "name": name,
            "bbox": (xmin, ymin, xmax, ymax),
            "area": (xmax - xmin) * (ymax - ymin),
            "width": xmax - xmin,
            "height": ymax - ymin,
            "center": ((xmin + xmax) / 2, (ymin + ymax) / 2)
        })

    return image_filename, image_width, image_height, objects

def determine_location(center, image_width, image_height):
    x, y = center
    horizontal = "left" if x < image_width / 3 else "right" if x > 2 * image_width / 3 else "center"
    vertical = "top" if y < image_height / 3 else "bottom" if y > 2 * image_height / 3 else "center"
    return f"{vertical}-{horizontal}"

def is_phone_in_corner(phone_bbox, image_width, image_height, margin=1):
    xmin, ymin, xmax, ymax = phone_bbox
    near_left = xmin <= margin
    near_right = xmax >= image_width - margin
    near_top = ymin <= margin
    near_bottom = ymax >= image_height - margin

    if (near_left and near_top) or (near_left and near_bottom) or (near_right and near_top) or (near_right and near_bottom):
        return True
    return False

def calculate_phone_metrics(objects, image_width, image_height):
    phones = [obj for obj in objects if obj['name'] == 'cellphone']
    hands = [obj for obj in objects if obj['name'] == 'hand']

    phone_metrics = []
    for phone in phones:
        in_hand = False
        location = determine_location(phone["center"], image_width, image_height)
        in_corner = is_phone_in_corner(phone["bbox"], image_width, image_height)

        # Check if phone is in hand (IoU calculation)
        for hand in hands:
            phone_bbox = phone["bbox"]
            hand_bbox = hand["bbox"]

            ixmin = max(phone_bbox[0], hand_bbox[0])
            iymin = max(phone_bbox[1], hand_bbox[1])
            ixmax = min(phone_bbox[2], hand_bbox[2])
            iymax = min(phone_bbox[3], hand_bbox[3])

            inter_area = max(0, ixmax - ixmin) * max(0, iymax - iymin)
            union_area = phone["area"] + hand["area"] - inter_area

            iou = inter_area / union_area if union_area > 0 else 0
            if iou > 0.01:
                in_hand = True
                break

        phone_metrics.append({
            "bbox": phone["bbox"],
            "area": phone["area"],
            "width": phone["width"],
            "height": phone["height"],
            "center": phone["center"],
            "location": location,
            "in_hand": in_hand,
            "in_corner": in_corner,
            "image_width": image_width,
            "image_height": image_height
        })

    return phone_metrics

def process_annotations(annotation_folder, output_csv_path):
    csv_data = []
    
    for xml_file in os.listdir(annotation_folder):
        if xml_file.endswith('.xml'):
            file_path = os.path.join(annotation_folder, xml_file)
            image_filename, image_width, image_height, objects = parse_voc_xml(file_path)
            phone_metrics = calculate_phone_metrics(objects, image_width, image_height)

            for phone in phone_metrics:
                csv_data.append({
                    "filename": annotation_folder + '/' + image_filename,
                    "bbox": phone["bbox"],
                    "area": phone["area"],
                    "width": phone["width"],
                    "height": phone["height"],
                    # "center": phone["center"],
                    "location": phone["location"],
                    "in_hand": phone["in_hand"],
                    "in_corner": phone["in_corner"],
                    "image_width": phone["image_width"],
                    "image_height": phone["image_height"]
                })

    with open(output_csv_path, mode='w', newline='') as csv_file:
        fieldnames = ["filename", "bbox", "area", "width", "height", "location", "in_hand", "in_corner", "image_width", "image_height"]
        writer = csv.DictWriter(csv_file, fieldnames=fieldnames)

        writer.writeheader()
        for row in csv_data:
            writer.writerow(row)

annotation_folder = "/home/ajeet/codework/datasets/Training_Validation_Dataset_2024_Ajeet-20241107T105545Z-001/Training_Validation_Dataset_2024_Ajeet/test_2021/2021/july/clients_equalized_jpg_xml"
output_csv_path = "/home/ajeet/codework/visionworkajeet/models/my_visionwork/models/clip/phone/train_phone_data_distribution/july.csv"
process_annotations(annotation_folder, output_csv_path)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

csv_path = '/home/ajeet/codework/visionworkajeet/models/my_visionwork/models/clip/phone/train_phone_data_distribution/train_merged_file.csv'  # Replace with your CSV file path
df = pd.read_csv(csv_path)

df['bbox'] = df['bbox'].apply(lambda x: eval(x))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(df['width'], df['height'], c='blue', alpha=0.6)
axes[0].set_xlabel('Phone Width')
axes[0].set_ylabel('Phone Height')
axes[0].set_title('Phone Width vs Height')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].hist(df['width'], bins=10, color='skyblue', edgecolor='black')
axes[0].set_xlabel('Phone Width')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Phone Widths')

axes[1].hist(df['height'], bins=10, color='lightgreen', edgecolor='black')
axes[1].set_xlabel('Phone Height')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Phone Heights')

axes[2].hist(df['in_hand'].apply(lambda x: 1 if x else 0), bins=2, color='orange', edgecolor='black')
axes[2].set_xlabel('In Hand (1=True, 0=False)')
axes[2].set_ylabel('Frequency')
axes[2].set_title('In Hand Status Distribution')

plt.tight_layout()
plt.show()


In [ ]:
width_mean = df['width'].mean()
width_std = df['width'].std()
print(f"width_mean:{width_mean}")
print(f"width_mean:{width_std}")

In [ ]:
height_mean = df['height'].mean()
height_std = df['height'].std()
print(f"width_mean:{height_mean}")
print(f"width_mean:{height_std}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

csv_path = '/home/ajeet/codework/visionworkajeet/models/my_visionwork/models/clip/phone/train_phone_data_distribution/train_merged_file.csv'  # Replace with your CSV file path
data = pd.read_csv(csv_path)
plt.figure(figsize=(10, 6))
plt.scatter(data['width'], data['height'], c='blue', alpha=0.5)
plt.title('Phone Width vs. Height')
plt.xlabel('Width')
plt.ylabel('Height')
plt.grid(True)
plt.show()

in_hand_data = data['in_hand'].value_counts()
in_corner_data = data['in_corner'].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].bar(in_hand_data.index.astype(str), in_hand_data.values, color=['green', 'red'])
axes[0].set_title('In Hand Distribution')
axes[0].set_xlabel('In Hand')
axes[0].set_ylabel('Count')
axes[1].bar(in_corner_data.index.astype(str), in_corner_data.values, color=['orange', 'blue'])
axes[1].set_title('In Corner Distribution')
axes[1].set_xlabel('In Corner')
axes[1].set_ylabel('Count')
plt.tight_layout()
plt.show()